# Tutorial: Hierarchical Workflow with AgentGrid SDKs
- Author: Shridhar Kini
- To Securely Run: `jupyter notebook password` to generate onetime password for secure access
- To Run at root directory of the repo: `jupyter notebook --allow-root --port 9999 --ip=0.0.0.0` 
- To Clear Outputs: Use `jupyter nbconvert --clear-output --inplace 8_Hierarchical_Pattern_Workflow_demo.ipynb` 

This walkthrough demonstrates how to build and orchestrate a hierarchical workflow of agents using the **AgentGrid SDKs** and **AgentGrid Workflow Engine**. We focus on a **Software Product Lifecycle Workflow**, where a CEO initiates a project and a Chief of Staff coordinates with specialized Team Leads (Marketing, Financial, Architecture, Development, Testing) and 12+ sub-agents in functional nested workflows.

## 1. Directory Structure
Understanding the files in this demonstration:
* **`nodes/`**: Contains the Python code for all 19 agents (`agent_ceo.py`, `agent_cos.py`, team lead agents, and specialist junior/senior agents).
* **`spec/`**: Contains agent registration JSON specifications (`agent_workflow_ceo.json`, etc.) and bash scripts for agent-level operations (registering, deploying, removing, unregistering).
* **`deploy_run/`**: Contains sub-workflow specifications (`workflow_spec-marketing.json` etc.), the master orchestrator spec (`workflow-final.json`), the Chief of Staff dynamic router spec (`workflow-cos.json`), the Streamlit dashboard code (`streamlit_app.py`, `structure`), and the bash scripts for workflow orchestration and inference.

#### Hierarchical Workflow Diagram
![Hierarchical Workflow](8_hierarchical_agentic_pattern.png)

## 2. Register and Build Individual Agents
Before agents can be deployed, they must be registered to the system and packaged as Docker containers.

### Register the agents

In [ ]:
%%bash
(
  cd ../spec/
  bash register_agents.sh
)

### Build the docker images and push them to the repository

In [ ]:
%%bash
(
  cd ../../
  bash build_hierarchical_workflow_and_push.bash
)

## 3. Deploy Agents
Once the images are built and pushed, deploy the agents into the cluster so they are actively waiting for tasks.

### Deploy the agents into the cluster

In [ ]:
%%bash
(
  cd ../spec/
  bash deploy_agents.sh
)

### Verify that the agents are running

In [ ]:
!kubectl get pods -n agents

## 4. Register and Deploy the Workflow
The `workflow-final.json`, `workflow-cos.json`, and functional sub-workflows define how the outputs of one agent map dynamically to the inputs of the next agent or nested sub-workflow. We need to register these specifications and deploy the master workflow.

### Register the workflow

In [ ]:
%%bash
(
  bash register_workflow.sh
)

### Deploy the workflow to establish connections
This creates a central coordinator for the workflow which acts like a Controller coordinating between different agents based on the workflow defined. It provides API URLs for interacting with inputs or tasks.

In [ ]:
%%bash
(
  bash deploy_workflow.sh
)

### Verify that the workflow pods are running

In [ ]:
!kubectl get pods -n workflows

## 5. Run the Workflow Demo
With everything deployed, start the Streamlit Dashboard to visualize the flow, and feed the initial payload into the first agent.

### Run the Streamlit app in the background

In [ ]:
%%bash
(
  source ../../../agent_codes/venv/bin/activate
  nohup bash run_streamlit_app.bash > streamlit.log 2>&1 &
  echo "Dashboard starting in background..."
)

### Trigger the inference by sending a project idea to the workflow inbox

In [ ]:
%%bash
(
  bash workflow_input.sh
)

## 6. Clean Up and Destroy Resources
Once the demonstration is complete, cleanly tear down the resources.

### Remove workflow connections and coordinator pods

In [ ]:
%%bash
(
  bash remove_workflow.sh
)

### Unregister the workflows entirely

In [ ]:
%%bash
(
  bash unregister_workflow.sh
)

### Remove the agent pods

In [ ]:
%%bash
(
  cd ../spec
  bash remove_agents.sh
)

### Unregister the individual agents from the AgentGrid system

In [ ]:
%%bash
(
  cd ../spec
  bash unregister_agents.sh
)